# Lesson 3.3 — Open-loop Loss vs Closed-loop Success

本课把前两课的量收在一起：3.1 说明"离线指标好 ≠ 闭环成功"，3.2 说明"误差如何在时间上被放大"。这一课问的是**评测本身**：既然两者不同，一个 robot policy 到底应该被怎么衡量？

对应 `docs/roadmap_v3.md` 的 Lesson 3 验收方向；更完整的 evaluation 讨论会在 Lesson 20（Offline ≠ Closed-loop）展开。

## 本课目标

学完本课后应该能：

1. 说明 offline action prediction loss 到底测量了什么；
2. 说明为什么低 MSE 不能保证任务完成；
3. 说明为什么 embodied agent 必须做 closed-loop evaluation；
4. 设计一个包含多种子成功率、鲁棒性测试与失败分类的评测；
5. 判断一篇论文报告的数字是否足以支持它的结论。

## 1. MSE 到底在衡量什么

给定 expert 数据 $D_E=\{(o_t,a_t)\}$，MSE 衡量的是

$$
L_\text{MSE}=\frac{1}{N}\sum_{t=1}^{N}\big\lVert \pi_\theta(o_t)-a_t\big\rVert_2^2
\qquad\text{其中 } o_t\sim D_E
$$

一句话：**在这个 observation 下，policy 的 action 接近 expert 的 action**。注意"在 expert 的 observation 下"这个前提——它是整个指标意义的边界。

但是注意，它没有回答：

- 下一步 state 是什么？
- task 是否完成？
- 是否抓住 cube？
- 是否稳定？

也就是说，MSE 是：

$$
\text{Action imitation metric}
$$

不是：

$$
\text{Task completion metric}
$$

两者的区别不是精度高低，而是**被测对象不同**：前者测函数在给定输入上的输出，后者测闭环系统在一段时间上的行为。

## 2. Open-loop evaluation（离线评测）

做法：**不让 policy 影响未来的 observation**，输入永远由 expert 提供（等价于监督学习里的 teacher forcing）：

```text
expert trajectory:  o_0 → o_1 → o_2 → …
                     │     │     │
policy               π_θ   π_θ   π_θ
                     ↓     ↓     ↓
prediction          â_0   â_1   â_2
compare             a_0*  a_1*  a_2*
```

每一步都是独立的：即使第 $t$ 步预测错了，第 $t+1$ 步的输入仍然是 expert 的 $o_{t+1}$，**误差不会传下去**。这就是它"稳定、可比较"的原因，也是它看不见累积误差的原因。

关键：

**输入永远来自 expert。**

$$
o_t\sim D_E \qquad\text{(expert state distribution)}
$$

数学：

$$
o_t\sim D_E,\qquad \hat a_t=\pi_\theta(o_t)
$$

逐点比较 prediction 与 target：

$$
\ell_t=\big\lVert \hat a_t-a_t^{\star}\big\rVert^2
$$

整条轨迹上取平均，就是 §1 定义的 $L_\text{MSE}$。

### 优点

- 快、便宜、稳定：不需要 simulator 或真机，也不需要 rollout；
- 可以比较模型：同一份 held-out 数据上，不同架构与超参能公平对比；
- 因此 VLA benchmark 常报告 action prediction error。

### 缺点

它隐藏了核心问题：**policy 自己产生的错误 state**。$D_E$ 里没有这些 state，所以 policy 在那些地方表现再差，offline 指标也不会变化。这正是 3.2 的 covariate shift——**误差不在被测量的分布上**。

## 3. Closed-loop evaluation（在线评测）

让 policy 真正控制环境：

流程：

```text
reset environment
      ↓
observe o_t
      ↓
policy π_θ
      ↓
action a_t
      ↓
env.step(a_t)
      ↓
new observation o_{t+1}
      ↓
repeat
```

数学：

$$
o_{t+1}=f\big(o_t,\ \pi_\theta(o_t)\big)
$$

注意：下一次的输入

$$
o_{t+1}
$$

**不是** expert 给你的。因此

$$
o_t\sim D_\pi \qquad\text{(policy state distribution)}
$$

$D_\pi\neq D_E$ 是常态而不是异常——这就是为什么 open-loop 与 closed-loop 会对同一个 policy 给出完全不同的结论。

## 4. 两种评测回答的是不同问题

| | Open-loop（offline） | Closed-loop（online） |
|---|---|---|
| 输入来自 | expert：$o_t\sim D_E$ | policy 自己：$o_t\sim D_\pi$ |
| 下一步 | 由 expert 提供，误差不传播 | $o_{t+1}=f(o_t,\pi_\theta(o_t))$，误差会传播 |
| 指标 | MSE / L1 / action accuracy | success rate、completion、return… |
| 回答的问题 | "会不会模仿" | "能不能完成任务" |
| 失败时暴露的信息 | 很少 | 失败在哪一步、属于哪一类 |

本仓库的真实例子（Lesson 2，2.8.7 / 2.8.8），四个数字来自同一份数据与同一个 checkpoint：

| 指标 | 数值 | 说明 |
|---|---|---|
| 验证 MSE（offline） | `0.2350` | 看起来是个小数，但它**劣于** mean-action baseline `0.1421` |
| held-out state 上的越界 action | `0.2%`（`1/608`） | 在 expert 分布上几乎"没问题" |
| 闭环需要 clip 的步数 | **`99.5%`** | 一进入 policy 自己的分布，输出立刻不可信 |
| 10 个 seed 的任务成功率 | **`0/10`** | 任务从未完成 |

**只有最后一个回答了"能不能完成任务"。**

## 5. 一个正确的 embodied AI evaluation 应该包含什么

以后读论文时，检查它报告了下面哪几类。

### 5.1 Offline metrics

- Action error：$L_\text{MSE}$、$L_1=|\hat a-a|$；
- 作用：判断**模型有没有学到 demonstration**；
- 局限：如 §4 所述，它不约束 $D_\pi$。

### 5.2 Online metrics

**Success rate（最重要）**

$$
SR=\frac{\#\{\text{successful episodes}\}}{\#\{\text{total episodes}\}}
$$

例如 10 条里成功 8 条：$SR=80\%$。

**Completion（长任务）**

$$
\text{completion}=\frac{\#\{\text{completed phases}\}}{\#\{\text{total phases}\}}
$$

例如 pick → place → close drawer 完成 2/3。

**Episode return**

$$
G=\sum_{t=0}^{T}\gamma^t r_t \qquad(\gamma\le 1)
$$

RL 中常用；在 imitation 里更多作为辅助诊断信号。

### 5.3 Robustness

机器人环境一定有变化，至少要分别测：

- **Object position**：train 时 cube 在 A，test 时在 B；
- **Sensor noise**：camera、joint state 加噪；
- **Disturbance**：例如人为移动 cube。

报告方式不是"能工作"，而是**每条扰动轴上的 $SR$ 与退化曲线**。

### 5.4 Efficiency 与 safety（最常被忽略）

- 到成功/失败所需的步数与时间；
- 越界 action 的比例（本仓库闭环评测里的 `99.5%` 就属于这一类诊断）；
- 碰撞、超力、危险动作的次数；
- 需要人类接管的频率（intervention rate）。

### 5.5 Calibration 与"安全拒绝"

如果 policy 带置信度，就要报告**置信度是否校准**：声称 80% 成功时是否真的约 80% 成功，以及它在不确定时会不会说"我不知道"。这与后续的 selective intervention 以及 `act / guide / ask / wait / recover / escalate` 直接相连。

### 5.6 Failure analysis

不能只写 $SR=40\%$，还要能回答"为什么失败"——见 §7 的 failure taxonomy。

## 6. Success rate 的统计细节：$0/10$ 到底等于什么

$SR$ 是一个**估计量**，只在有限条 episode 上测出来，因此必须报告不确定度。$K$ 条 episode 的经验成功率的二项标准误是

$$
\mathrm{SE}(\widehat{SR})=\sqrt{\frac{\widehat{SR}\,(1-\widehat{SR})}{K}}
$$

更有用的是**成功数为 0** 时的单侧置信上界（Clopper–Pearson）：

$$
SR_\text{upper}=1-(1-\alpha)^{1/K}
$$

代入 $\alpha=0.05,\ K=10$：$SR_\text{upper}=1-0.05^{0.1}\approx 26\%$。也就是说，**"0/10 成功"不等于"成功率是 0"**，它只排除了约 26% 以上的成功率。

这直接解释了 Lesson 2 为什么把 2.8.8 从 1 个 seed 扩到 10 个 seed：单条 episode 是 case study，不是 rate；而 10 条给出的也是一个较宽的上界。要得到可引用的成功率，需要**更多 episode × 多种子**，并同时报告失败分类。

还有一个必须写进论文的细节：**评测协议本身**——每 episode 多少步、reset 分布是什么、是否允许人类干预、time limit 是多少。Lesson 2 里默认的 `TimeLimit=50` 就曾把结论引向错误方向，必须先排除它。

## 7. Failure Taxonomy（非常重要）

以后做 research，你不能只写：

$$
SR=40\%
$$

还要知道：**为什么失败？**

同一个 $SR$ 可能来自完全不同的原因，而不同原因对应完全不同的修法——加数据、改架构、加历史、还是加安全层。

机器人失败分类。每一类都要有**可观察的判据**，否则无法归因：

| 失败类型 | 典型现象 | 可观察判据 |
|---|---|---|
| **Perception failure** | 看错：object pose wrong | 感知输出与 privileged state 的偏差（仿真里可查） |
| **Planning failure** | 不知道下一步：wrong subgoal | 子目标序列与任务结构不符 |
| **Control failure** | 动作执行错误：overshoot | 跟踪误差、越界/clip 比例 |
| **Distribution shift failure** | 训练没见过：unseen state | held-out state 的 $\lvert z\rvert$、最近邻距离 |
| **Recovery failure** | 犯错后无法恢复 | 失败后的重试成功率、是否长时间停滞 |

本仓库 Lesson 2 的失败属于第 4 类：held-out state 上 $\lvert z\rvert=13.21$、闭环 `99.5%` 的步需要 clip、reward 在第 20 步左右归零并保持到第 200 步。

## 8. 为什么 ACT / Diffusion / VLA 改善这个问题？

"误差累积"和"多峰动作"是 BC 的结构性问题，后来的方法从不同环节下手：

- 改**决策频率**（ACT）；
- 改**输出分布**（Diffusion Policy）；
- 改**条件信息**（VLA）。

### BC

问题：single action prediction——每个时刻一次独立决策，误差有 $T$ 次进入回路的机会：

$$
o_t \rightarrow a_t
$$

### ACT

预测 future action sequence（action chunk）：

$$
o_t \rightarrow (a_t,a_{t+1},\dots,a_{t+k})
$$

优势：减少每一步重新决策造成的误差累积（决策点从 $T$ 降到约 $T/k$），且 chunk 内部动作时间一致。注意它**降低**误差注入的频率，但不消除分布偏移。

### Diffusion Policy

学习的是动作的**分布**，而不是唯一答案：

$$
p_\theta(a\mid o) \quad\text{或}\quad p_\theta(A_t\mid o_t)
$$

不是 "move left 5cm" 这一个答案，而是 trajectory distribution。这样当同一个 observation 下存在多个都合理的动作（multimodal）时，不会像 MSE 那样把它们平均成一个谁都执行不了的动作。

### VLA

进一步加入 vision、language 与 task context：

```text
"pick the red cube"  +  current observation  +  memory
                            ↓
                     action (chunk)
```

它改变的是**条件信息**：language 给出任务身份，vision 给出物体与场景，memory 给出"之前发生过什么"。这解决的是"泛化到新任务/新物体"，而不是直接消除误差累积——两者是不同的问题，需要不同的评测（§5 的 robustness 与 §7 的 failure taxonomy）。

## 小结

Open-loop evaluation 测量的是：

> "Can the policy imitate expert actions on expert states?"

Closed-loop evaluation 测量的是：

> "Can the policy complete the task when controlling the environment?"

**这是两个不同的问题。** 一个 policy 可以有很低的 action prediction error，却几乎无法完成任务，因为：

1. policy 的 action 改变了未来的 observation：$o_{t+1}=f(o_t,\pi_\theta(o_t))$；
2. 部署时的 state 分布 $D_\pi$ 与训练时的 $D_E$ 不同（covariate shift，见 3.2）；
3. 小误差沿 horizon 累积（compounding error，见 3.2 的 $e_{t+1}=(1-0.1k)e_t$）。

因此 embodied AI 的评测必须包含：

- offline metrics：模型是否学到 demonstration；
- closed-loop success：能否完成任务，带多种子与置信区间；
- robustness tests：扰动轴上的退化；
- failure analysis：失败类型与可观察判据。

**本仓库的实例**：验证 MSE `0.2350`（且劣于 mean-action baseline `0.1421`）、离线越界 `0.2%`、闭环需 clip `99.5%`、10 seed 成功率 `0/10`（95% 单侧上界约 `26%`）。四个数字里只有最后一个在回答"能不能完成任务"。